# Сны грунта: сеть предлагает прочтение, грунт проверяет

Ученик (`rows.py` + `records.py`) находит смысл предложений новой задачи поиском: от минут до часов на задачу, и почти всё это время уходит на прочтения, далёкие от верного. Если прочтения предложений даны, та же задача выучивается за десятую долю секунды. Здесь их угадывает сеть, выращенная на «снах» грунта — до всяких слов:

- **сны** (`relive.py`): задачи, которые ученик уже выучил целиком, проживаются заново — в своих рассказах и в новых, которых не было, — и в вариациях: одно-два предложения прочитаны иначе, ответы пересчитаны грунтом. Автора у таких задач нет, их сколько угодно;
- **сеть** (`guess.py`): трансформер читает рассказы новой задачи (слов не видит: только какие виды предложений есть, какие вещи в них стоят и какими были ответы) и для каждого вида предложений называет прочтения, которые стоит пробовать первыми;
- **грунт** проверяет предложенное точно, как любую догадку, и чинит: по очереди пробует первые варианты для каждого вида предложений. Где ничего не подошло — обычный поиск;
- **та же сеть без грунта**: вторым выходом она сама отвечает на вопросы нового рассказа, имея перед собой только предыдущие рассказы задачи с ответами. Это то, что сеть, выращенная на снах грунта, может сама; первое — что она может, когда исполняет и проверяет грунт.

Что измеряется: наборы H, I, J и K (48 задач других авторов) в обучении сети не участвуют. На каждой задаче — её первые 20 рассказов: объясняет ли первое же предложение сети все вопросы, объясняет ли после починки, и что это прочтение даёт на следующих сорока рассказах; рядом — память слово в слово (`memory.py`) и ответы сети без грунта на тех же сорока рассказах.

Как запускать: среда выполнения T4 (или лучше), «Выполнить всё». Обучение сети — `MX_MINUTES` минут (по умолчанию 90); оборванный запуск при повторном выполнении продолжается с сохранённого места. Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки. Нужен вывод ячеек с итогами (строки, начинающиеся с «tasks» и «all the tasks», и таблица в конце).

In [ ]:
import os, subprocess, sys, json, re, time, glob
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
MINUTES = float(os.environ.get('MX_MINUTES', 0.5 if QUICK else 90))
WORK = ROOT + '/mx-dreams'
os.makedirs(WORK, exist_ok=True)
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
os.environ['PYTHONHASHSEED'] = '0'
!python -u {M5}/colab/mind.py --work {WORK} --fetch | tail -1
for name in 'ABCDEFGHIJK':
    !python {M5}/newtasks.py --set {name} --out {WORK}/new/{name} | tail -1

## Задачи, как они были поняты

Из готовых результатов ученика (`results/rows/*.txt`) берутся прочтения предложений; программы вопросов грунт находит заново одной подгонкой — секунды на все задачи.

In [ ]:
R = M5 + '/results/rows'
specs = [f'{WORK}/tasks_1-20_v1-2/en:{R}/rerun_en.txt'] + [f'{WORK}/new/{x}:{R}/rerun_{x}.txt' for x in 'ABCDEFG']
specs += [f'{WORK}/new/H:{R}/bridge_H_sure.txt', f'{WORK}/new/I:{R}/bridge_I.txt', f'{WORK}/new/J:{R}/bridge_J.txt', f'{WORK}/new/K:{R}/bridge_K.txt']
!cd {M5} && python relive.py --tasks {','.join(specs)} --out {WORK}/lived.pkl | tail -1

## Сеть учится на снах и пробуется на чужих задачах

Наборы H, I, J и K отложены: сны строятся только из задач bAbI и наборов A–G. В конце — по строке на каждую отложенную задачу и два итога: по задачам, которые ученик выучил поиском целиком, и по остальным.

In [ ]:
size = '--width 64 --layers 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --look 300 --every 20 --few 4' if QUICK else '--width 384 --layers 8 --heads 8 --batch 24 --every 200'
!cd {M5} && python -u guess.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {size} --out {WORK}/guess.pt 2>&1 | tee -a {WORK}/guess.txt | grep -v "^step [0-9]*[1-9]00,"

## Необязательно: петля вместо высокой укладки

Та же сеть, но слоёв два, и они проходятся четыре раза подряд с теми же весами: вычислений столько же, сколько у восьми слоёв, весов вчетверо меньше. Отвечает на вопрос, что важнее для рассуждения над состоянием — число разных слоёв или число шагов. Включается отдельно: `%env MX_LOOPS=1` до первой ячейки (ещё столько же минут обучения).

In [ ]:
if os.environ.get('MX_LOOPS') or QUICK:
    loop = '--width 64 --layers 1 --loops 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --look 300 --every 20 --few 4' if QUICK else '--width 384 --layers 2 --loops 4 --heads 8 --batch 24 --every 200'
    !cd {M5} && python -u guess.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {loop} --out {WORK}/guess_loop.pt 2>&1 | tee -a {WORK}/guess_loop.txt | grep -v "^step [0-9]*[1-9]00,"
else:
    print('пропущено: чтобы сравнить с петлёй, выполните %env MX_LOOPS=1 до первой ячейки')

## Сеть отвечает сама: грунт через топологию

То же, но без грунта вообще (`mind.py`): сеть читает рассказы новой задачи слева направо и на каждом вопросе говорит ответ; слов по-прежнему не видит, имена видов предложений и вещей раздаются заново в каждой задаче, так что смысл можно взять только из предыдущих рассказов. Сравниваются сборки одного размера и времени обучения:

- **обычная** — слои один над другим;
- **с памятью записей** (`--names 2`) — каждое утверждение рассказа остаётся записью (вид предложения и его вещи), и в любом месте сеть может вспомнить последние записи, где названа вещь. Встроено только это: имя находится точно (метки слов не обучаются), последнее упоминание — первым, ответ можно взять словом из вспомненной записи. Какое имя спросить, у какого вида предложений, что взять из записи и когда пойти по цепочке — выучивается;
- необязательно **петля** (`%env MX_LOOPS=1`) — два слоя, пройденные четыре раза, с той же памятью записей.

Проверка — те же отложенные наборы H, I, J и K: перед сетью первые рассказы задачи с ответами, ответить надо на вопросы следующих; затем каждой задаче даётся ещё 300 малых шагов на её собственных двадцати рассказах, и проверка повторяется. Каждая сборка учится `MX_MINUTES` минут.

In [ ]:
size = '--width 64 --layers 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --every 20 --few 3 --tune 3' if QUICK else '--width 384 --layers 8 --heads 8 --batch 24 --every 200 --tune 300'
builds = [('с памятью записей', 'mind_names', '--names 2'), ('обычная', 'mind_plain', '')]
if os.environ.get('MX_LOOPS') and not QUICK:
    builds.append(('петля с памятью записей', 'mind_loop', '--names 2 --loops 4 --layers 2'))
for label, name, more in builds:
    print('=====', label)
    !cd {M5} && python -u mind.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {size} {more} --out {WORK}/{name}.pt 2>&1 | tee -a {WORK}/{name}.txt | grep -E "weights|^tasks|^all the tasks|^step [0-9]*000,"

## Ученик с сетью и без неё

Те же отложенные задачи целиком, как в слепых прогонах: 20 рассказов и до 20 запрошенных, проверка на тесте. С сетью (`--guess --fast`): если предложенное прочтение объясняет все вопросы первых рассказов и запрошенных, оно берётся без поиска; иначе поиск идёт как обычно, начиная с предложенного. Без сети — числа слепых прогонов из `results/rows/bridge_*.txt`. Поиск считает на процессоре и на двух ядрах Colab занимает несколько часов, поэтому часть включается отдельно: `%env MX_FULL=1` до первой ячейки.

In [ ]:
def table(path):
    out = {}
    for l in open(path):
        m = re.match(r'task +(\d+):.*\((\d+) s, ', l)
        if m:
            k, secs, quick = int(m.group(1)), int(m.group(2)), 'taken at once' in l
        m = re.match(r'deep +(\d+): (\d+)/(\d+)', l)
        if m:
            out[int(m.group(1))] = (100 * int(m.group(2)) / int(m.group(3)), secs, quick)
    return out

if os.environ.get('MX_FULL') or QUICK:
    !cd {M5} && python memory.py --out {WORK}/memory_AG.json results/rows/rerun_*.txt | tail -1
    sets = {'H': 'bridge_H_sure.txt'} if QUICK else {'H': 'bridge_H_sure.txt', 'I': 'bridge_I.txt', 'J': 'bridge_J.txt'}
    halves = ('1',) if QUICK else ('1,3,5,7,9,11', '2,4,6,8,10,12')
    for name, old in sets.items():
        jobs = [subprocess.Popen(f"cd {M5} && {sys.executable} -u rows.py --data {WORK}/new/{name} --ask 20 --limit {40000 if QUICK else 300000} --tasks {h} --show "
                                 f"--memory {WORK}/memory_AG.json --guess {WORK}/guess.pt --fast > {WORK}/with_{name}_{i}.txt 2>&1", shell=True) for i, h in enumerate(halves)]
        for j in jobs:
            j.wait()
        new, was = {}, table(f'{R}/{old}')
        for i in range(len(halves)):
            new.update(table(f'{WORK}/with_{name}_{i}.txt'))
        print(f'набор {name}: задача: верно % (время, с) с сетью | без сети')
        for k in sorted(new):
            w = was.get(k)
            print(f"  {name}{k}: {new[k][0]:.1f} ({new[k][1]} с{', взято сразу' if new[k][2] else ''}) | " + (f"{w[0]:.1f} ({w[1]} с)" if w else 'не досчиталась'))
        ks = sorted(k for k in new if k in was)
        print(f"  по задачам, досчитавшимся в обоих прогонах: среднее {sum(new[k][0] for k in ks) / len(ks):.1f} | {sum(was[k][0] for k in ks) / len(ks):.1f}; "
              f"до 95 % и выше {sum(new[k][0] >= 95 for k in ks)} | {sum(was[k][0] >= 95 for k in ks)} из {len(ks)}; "
              f"взято сразу {sum(new[k][2] for k in ks)}; время {sum(new[k][1] for k in ks)} | {sum(was[k][1] for k in ks)} с")
else:
    print('пропущено: чтобы посчитать ученика целиком, выполните %env MX_FULL=1 до первой ячейки')